In [ ]:
# ga_optimize_hybrid.py
from __future__ import annotations
import random
import math
from functools import lru_cache
from datetime import datetime, timezone
from pathlib import Path
import csv

from deap import base, creator, tools, algorithms  # noqa: F401 (algorithms unused but fine)

# ---- your model ----
from model_system import run_hybrid_summary

# =========================
# Search space & utilities
# =========================
# Bounds (tune if you have better priors)
BOUNDS = [
    (50_000.0, 300_000.0),   # desired_array_size [kWdc]
    (50.0, 200.0),           # P_ref [MWe]
    (0.0, 16.0),             # t_TES_hours [h]
    (1.0, 5.0),              # solarm [-]
]

# Coarse grid helps stability + cache hit-rate
RESOLUTION = [500.0, 1.0, 0.5, 0.1]  # rounding step for each variable

def clamp(val, lo, hi):
    return max(lo, min(hi, val))

def repair(ind):
    """Keep individuals inside bounds & snap to a grid for stability + caching."""
    for i, (lo, hi) in enumerate(BOUNDS):
        step = RESOLUTION[i]
        ind[i] = clamp(round(ind[i] / step) * step, lo, hi)
    return ind

# =========================
# Objective extraction
# =========================
def _get_num(d: dict, *cand_keys, default=None):
    for k in cand_keys:
        if k in d and d[k] is not None:
            try:
                return float(d[k])
            except Exception:
                pass
    return default

# -------------------------
# Trial logging (ALL calls)
# -------------------------
_TRIAL_LOG = []
_CURRENT_GEN = -1   # updated in main() loop before evaluations
_SEEN_PARAMS = set()  # tracks evaluated (das, pref, th, sm)

def _log_trial(gen, das, pref, th, sm,
               ppa, cf, fit_ppa, fit_cf,
               status, note="",
               lcoe=None, annual_rev=None, cf_pb=None,
               pv_mwh=None, csp_mwh=None):
    _TRIAL_LOG.append({
        "gen": gen,
        "desired_array_size": das,
        "P_ref": pref,
        "t_TES_hours": th,
        "solarm": sm,
        "PPA_USD_per_MWh": ppa,
        "CF_hybrid": cf,
        "CF_pb": cf_pb,
        "LCOE": lcoe,
        "Annual_Revenue_USD": annual_rev,
        "PV_to_Grid_MWh": pv_mwh,
        "CSP_to_Grid_MWh": csp_mwh,
        "fitness_PPA": fit_ppa,     # same as PPA unless penalized
        "fitness_CF": fit_cf,       # same as CF unless penalized
        "status": status,
        "note": note,
        "timestamp": datetime.now(timezone.utc).isoformat()
    })

def _write_trials_csv(path: str | Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    keys = [
        "gen","desired_array_size","P_ref","t_TES_hours","solarm",
        "PPA_USD_per_MWh","CF_hybrid","CF_pb","LCOE",
        "Annual_Revenue_USD","PV_to_Grid_MWh","CSP_to_Grid_MWh",
        "fitness_PPA","fitness_CF","status","note","timestamp"
    ]
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        for row in _TRIAL_LOG:
            writer.writerow(row)

# =========================
# Cached evaluator
# =========================
@lru_cache(maxsize=2048)
def _evaluate_cached(desired_array_size, P_ref, t_TES_hours, solarm):
    """
    Evaluate one design.
    Objective: minimize PPA, maximize CF_hybrid.
    Returns (fitness_PPA, fitness_CF, result_dict, status)
    """
    try:
        res = run_hybrid_summary(
            desired_array_size=desired_array_size,
            P_ref=P_ref,
            t_TES_hours=t_TES_hours,
            solarm=solarm,
        )
        # Some implementations might return (dict,) — normalize:
        if isinstance(res, tuple) and len(res) == 1 and isinstance(res[0], dict):
            res = res[0]

        if not isinstance(res, dict):
            # Unknown structure -> hard penalty
            PENALTY_MAX, PENALTY_MIN = 1e9, -1e9
            return (PENALTY_MAX, PENALTY_MIN, {}, "penalty_bad_return")

        ppa = _get_num(res, "PPA", "PPA_USD_per_MWh", default=None)
        cf  = _get_num(res, "CF_hybrid", "CF", default=None)

        PENALTY_MAX, PENALTY_MIN = 1e9, -1e9
        if ppa is None or not math.isfinite(ppa):
            return (PENALTY_MAX, PENALTY_MIN, res, "penalty_missing_ppa")
        if cf is None or not math.isfinite(cf):
            return (ppa, PENALTY_MIN, res, "penalty_missing_cf")

        # Fitness = (PPA, CF) with weights=(-1, +1)
        return (ppa, cf, res, "ok")

    except Exception as e:
        PENALTY_MAX, PENALTY_MIN = 1e9, -1e9
        return (PENALTY_MAX, PENALTY_MIN, {}, f"exception:{type(e).__name__}")

def evaluate(individual):
    """DEAP evaluate wrapper: rounds, caches, logs full results."""
    repair(individual)
    das, pref, th, sm = individual

    # round to evaluation grid (hashable for cache + dedup)
    das = round(das / RESOLUTION[0]) * RESOLUTION[0]
    pref = round(pref / RESOLUTION[1]) * RESOLUTION[1]
    th   = round(th   / RESOLUTION[2]) * RESOLUTION[2]
    sm   = round(sm   / RESOLUTION[3]) * RESOLUTION[3]

    key = (das, pref, th, sm)
    note = "repeat" if key in _SEEN_PARAMS else "new"
    _SEEN_PARAMS.add(key)

    fit_ppa, fit_cf, res_dict, status = _evaluate_cached(das, pref, th, sm)

    # pull extra fields for logging (if present)
    ppa   = res_dict.get("PPA", res_dict.get("PPA_USD_per_MWh"))
    cf    = res_dict.get("CF_hybrid", res_dict.get("CF"))
    lcoe  = res_dict.get("LCOE")
    rev   = res_dict.get("Annual_Revenue", res_dict.get("Annual_Revenue_USD"))
    pv_mw = res_dict.get("PV_to_Grid_MWh")
    csp_mw= res_dict.get("CSP_to_Grid_MWh")
    cf_pb = res_dict.get("CF_pb")

    _log_trial(
        gen=_CURRENT_GEN,
        das=das, pref=pref, th=th, sm=sm,
        ppa=ppa, cf=cf,
        fit_ppa=fit_ppa, fit_cf=fit_cf,
        status=status, note=note,
        lcoe=lcoe, annual_rev=rev, cf_pb=cf_pb,
        pv_mwh=pv_mw, csp_mwh=csp_mw
    )

    return (fit_ppa, fit_cf)

# ================
# DEAP boilerplate
# ================
# Two-objective: (min PPA, max CF)
try:
    creator.FitnessMin2
except AttributeError:
    creator.create("FitnessMin2", base.Fitness, weights=(-1.0, 1.0))
try:
    creator.Individual
except AttributeError:
    creator.create("Individual", list, fitness=creator.FitnessMin2)

toolbox = base.Toolbox()

def _rand_between(lo, hi):
    return random.uniform(lo, hi)

for i, (lo, hi) in enumerate(BOUNDS):
    toolbox.register(f"attr_{i}", _rand_between, lo, hi)

toolbox.register(
    "individual",
    tools.initCycle,
    creator.Individual,
    (toolbox.attr_0, toolbox.attr_1, toolbox.attr_2, toolbox.attr_3),
    n=1,
)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("evaluate", evaluate)

# SBX crossover + polynomial mutation for bounded reals
toolbox.register("mate", tools.cxSimulatedBinaryBounded,
                 low=[b[0] for b in BOUNDS], up=[b[1] for b in BOUNDS], eta=15.0)
toolbox.register("mutate", tools.mutPolynomialBounded,
                 low=[b[0] for b in BOUNDS], up=[b[1] for b in BOUNDS], eta=20.0, indpb=0.25)
toolbox.register("select", tools.selNSGA2)

def main(seed: int = 42,
         pop_size: int = 24,
         ngen: int = 30,
         cxpb: float = 0.85,
         mutpb: float = 0.25,
         log_csv: str | Path = "ga_trials.csv"):
    global _CURRENT_GEN
    random.seed(seed)

    pop = toolbox.population(n=pop_size)
    hof = tools.ParetoFront()  # non-dominated set

    # Stats (track objectives)
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("min_PPA", lambda fits: min(f[0] for f in fits))
    stats.register("best_CF", lambda fits: max(f[1] for f in fits))

    # Initial evaluation (NSGA-II requires valid fitness before first select)
    _CURRENT_GEN = 0
    for ind in pop:
        repair(ind)
        ind.fitness.values = toolbox.evaluate(ind)

    # Sort & crowding distance
    pop = toolbox.select(pop, len(pop))
    hof.update(pop)

    # Evolve
    for gen in range(1, ngen + 1):
        _CURRENT_GEN = gen

        # Variation
        offspring = tools.selTournamentDCD(pop, len(pop))
        offspring = [toolbox.clone(ind) for ind in offspring]

        for i in range(0, len(offspring), 2):
            if random.random() < cxpb and i + 1 < len(offspring):
                toolbox.mate(offspring[i], offspring[i + 1])
                repair(offspring[i]); repair(offspring[i + 1])
                if hasattr(offspring[i].fitness, "values"):
                    del offspring[i].fitness.values
                if hasattr(offspring[i + 1].fitness, "values"):
                    del offspring[i + 1].fitness.values

        for i in range(len(offspring)):
            if random.random() < mutpb:
                toolbox.mutate(offspring[i])
                repair(offspring[i])
                if hasattr(offspring[i].fitness, "values"):
                    del offspring[i].fitness.values

        # Evaluate invalid
        invalid = [ind for ind in offspring if not ind.fitness.valid]
        for ind in invalid:
            ind.fitness.values = toolbox.evaluate(ind)

        # Combine + select
        pop = toolbox.select(pop + offspring, pop_size)
        hof.update(pop)

        # Stats
        record = stats.compile(pop)
        print(f"Gen {gen:>3}/{ngen} | min PPA: {record['min_PPA']:.3f} USD/MWh | "
              f"best CF: {record['best_CF']:.4f}")

    # Report final Pareto front
    print("\n=== Pareto front (PPA, CF) ===")
    for ind in hof:
        ppa, cf = ind.fitness.values
        print(
            f"das={ind[0]:.0f} kWdc | P_ref={ind[1]:.1f} MWe | TES={ind[2]:.1f} h | SM={ind[3]:.1f} "
            f"| PPA={ppa:.2f} USD/MWh | CF={cf:.4f}"
        )

    # Best-by-PPA convenience
    best_by_ppa = min(hof, key=lambda x: x.fitness.values[0])
    ppa, cf = best_by_ppa.fitness.values
    print("\nBest-by-PPA candidate:")
    print(
        f"das={best_by_ppa[0]:.0f} kWdc | P_ref={best_by_ppa[1]:.1f} MWe | TES={best_by_ppa[2]:.1f} h | SM={best_by_ppa[3]:.1f} "
        f"| PPA={ppa:.2f} USD/MWh | CF={cf:.4f}"
    )

    # ---- write ALL trials to CSV ----
    _write_trials_csv(log_csv)
    print(f"\nSaved ALL trials to: {Path(log_csv).resolve()}")

    return pop, hof

if __name__ == "__main__":
    # NOTE (Windows): keep GA execution inside __main__ guard.
    main(
        seed=123,
        pop_size=40,    # lower if each evaluation is slow; raise for broader search
        ngen=20,        # increase for better convergence
        cxpb=0.9,
        mutpb=0.3,
        log_csv="ga_trials.csv"
    )


✅ Collector type set to: Luz LS-3
✅ Total DIRECT CAPEX    : 234.29 MUSD
✅ Total INDIRECT CAPEX  : 49.20 MUSD
✅ Total CAPEX           : 283.49 MUSD
✅ Total OPEX: 4.21 MUSD/year

--- Hybrid System Summary ---
desired_array_size       : 63,000.0000
P_ref                    : 63.0000
t_TES_hours              : 6.5000
PPA                      : 25.4730
CF_hybrid                : 0.3727
CF_pb                    : 0.4438
LCOE                     : 66.8335
Annual_Revenue           : 13,014,560.5347
PV_to_Grid_MWh           : 106,047.9217
CSP_to_Grid_MWh          : 220,420.9337
✅ Collector type set to: Luz LS-3
✅ Total DIRECT CAPEX    : 1043.38 MUSD
✅ Total INDIRECT CAPEX  : 219.11 MUSD
✅ Total CAPEX           : 1262.49 MUSD
✅ Total OPEX: 11.34 MUSD/year

--- Hybrid System Summary ---
desired_array_size       : 275,500.0000
P_ref                    : 56.0000
t_TES_hours              : 8.5000
PPA                      : 73.5526
CF_hybrid                : 0.5217
CF_pb                    : 0.3858
L